# Brain Age Prediction: Hybrid Multimodal Ensemble
Questo notebook implementa una logica di Ensemble Avanzata e Asimmetrica basata sulle performance empiriche:
1. **FLAIR (Peso 50%)**: Utilizza esclusivamente il *Best Model Singolo* (poiché il K-Fold Ensemble su FLAIR diluiva le performance).
2. **T1w (Peso 50%)**: Utilizza il *K-Fold Ensemble* (tutti e 5 i modelli), pesando ciascuna rete T1 per 1/10 (ovvero 5 * 1/10 = 1/2 totale).
3. Combina le probabilità sul Test Set incontaminato e genera i grafici di valutazione finale.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('/kaggle/working/SFCN')

In [ ]:
import os
import json
import glob
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Import dal repository
from dp_model.model_files.sfcn import SFCN
from dp_model import dp_utils as dpu

PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Dataset Custom (Multimodale)

In [ ]:
class BrainAgeDataset(Dataset):
    def __init__(self, data_dir, modality='FLAIR', is_train=False):
        self.data_dir = data_dir
        self.modality = modality 
        self.is_train = is_train
        self.subject_dirs = sorted(glob.glob(os.path.join(data_dir, "sub-*")))
        self.samples = []
        
        self.bin_range = [0, 70]
        self.bin_step = 1
        self.sigma = 1.0
        
        for subj_dir in self.subject_dirs:
            subj_id = os.path.basename(subj_dir)
            
            nii_path = os.path.join(subj_dir, f"{subj_id}_{self.modality}_MNI152_1mm.nii")
            if not os.path.exists(nii_path):
                nii_path = nii_path + ".gz"
                if not os.path.exists(nii_path):
                    if self.modality == 'T1w':
                        nii_path_alt = os.path.join(subj_dir, f"{subj_id}_T1_MNI152_1mm.nii.gz")
                        if os.path.exists(nii_path_alt):
                            nii_path = nii_path_alt
                        else:
                            continue
                    else:
                        continue
                    
            json_path = os.path.join(subj_dir, f"{subj_id}_participant_info.json")
            if not os.path.exists(json_path):
                continue
                
            with open(json_path, 'r') as f:
                info = json.load(f)
                
            participant_info = info.get("participant_info", {})
            age_cat_val = participant_info.get("age_scan")
            
            if age_cat_val is None:
                continue
            
            try:
                age_cat = int(age_cat_val) - 1
                true_age = 3 + age_cat * 5
                y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
            except:
                continue
            
            self.samples.append({
                "nii_path": nii_path,
                "label_vect": y,
                "true_age": true_age
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        dx, dy, dz = 0, 0, 0
        
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_data, label_vect, sample['true_age']

## 2. Preparazione dei Modelli e del Test Set

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/elenaschgor/dataset-2-t1-flair/ds004199_final/"

# --- CONFIGURAZIONE PATH MODELLI ---
# Assicurati di aver fatto l'upload dei pesi .pth su Kaggle e modifica questi percorsi!
PATH_MIGLIOR_FLAIR = "/kaggle/input/sfcn-pesi/sfcn_FLAIR_best_fold.pth"

PATHS_5_FOLD_T1 = [
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_1.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_2.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_3.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_4.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_5.pth"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}\n")

# Recupero l'indice esatto del Test Set usato in addestramento (garantito dal random_state=42)
dummy_dataset = BrainAgeDataset(KAGGLE_DATA_DIR, modality='FLAIR', is_train=False)
dataset_size = len(dummy_dataset)

if dataset_size > 0:
    all_ages = [sample['true_age'] for sample in dummy_dataset.samples]
    all_indices = np.arange(dataset_size)
    _, test_idx = train_test_split(all_indices, test_size=0.10, random_state=42, stratify=all_ages)
    
    # Creazione DataLoader separati per T1 e FLAIR (stessi pazienti, stesse etichette, immagini diverse)
    test_dataset_flair = torch.utils.data.Subset(BrainAgeDataset(KAGGLE_DATA_DIR, modality='FLAIR', is_train=False), test_idx)
    test_loader_flair = DataLoader(test_dataset_flair, batch_size=1, shuffle=False)
    
    test_dataset_t1 = torch.utils.data.Subset(BrainAgeDataset(KAGGLE_DATA_DIR, modality='T1w', is_train=False), test_idx)
    test_loader_t1 = DataLoader(test_dataset_t1, batch_size=1, shuffle=False)
    
    # --- CARICAMENTO RETI IN RAM ---
    print("Caricamento Miglior Rete FLAIR...")
    model_flair = SFCN(output_dim=70)
    model_flair.load_state_dict(torch.load(PATH_MIGLIOR_FLAIR, map_location=device))
    model_flair.to(device)
    model_flair.eval()
    
    print("Caricamento 5 Reti T1 (K-Fold)...")
    models_t1 = []
    for p in PATHS_5_FOLD_T1:
        m = SFCN(output_dim=70)
        m.load_state_dict(torch.load(p, map_location=device))
        m.to(device)
        m.eval()
        models_t1.append(m)

## 3. Valutazione e Media Pesata Asimmetrica

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print("   AVVIO HYBRID ENSEMBLE SUL TEST SET")
    print("==============================================")
    
    ensemble_errors = []
    all_true_ages = []
    all_ensemble_preds = []
    
    bin_centers = np.arange(0, 70, 1)
    
    with torch.no_grad():
        # Zippiamo i due DataLoader per iterare in parallelo sui pazienti (FLAIR e T1 dello stesso soggetto)
        for (inputs_flair, _, true_age), (inputs_t1, _, _) in zip(test_loader_flair, test_loader_t1):
            inputs_flair = inputs_flair.to(device)
            inputs_t1 = inputs_t1.to(device)
            true_age_val = true_age.item()
            
            # --- 1. PROBABILITÀ FLAIR (Peso 1/2) ---
            out_flair = model_flair(inputs_flair)[0].view(1, -1)
            prob_flair = torch.exp(out_flair).cpu().numpy()
            
            # --- 2. PROBABILITÀ T1 (5 Modelli, ognuno pesa 1/10) ---
            probs_t1_list = []
            for m in models_t1:
                out_t1 = m(inputs_t1)[0].view(1, -1)
                prob_t1 = torch.exp(out_t1).cpu().numpy()
                probs_t1_list.append(prob_t1)
            
            # La somma dei 5 array divisa per 5 equivale matematicamente a pesare ogni modello 1/5.
            # Più avanti dimezzeremo questo valore per fargli valere 1/10 sul totale (Ensemble T1 al 50%).
            prob_t1_ensemble = np.mean(probs_t1_list, axis=0)
            
            # --- 3. L'INCASTRO FINALE (MEDIA PESATA ASIMMETRICA) ---
            # prob_final = (1/2 * prob_flair) + (1/2 * prob_t1_ensemble)
            # E siccome prob_t1_ensemble = (m1+m2+m3+m4+m5)/5, la riga sotto equivale esattamente a:
            # prob_final = 1/2*flair + 1/10*m1 + 1/10*m2 + 1/10*m3 + 1/10*m4 + 1/10*m5
            final_hybrid_prob = (0.5 * prob_flair) + (0.5 * prob_t1_ensemble)
            
            # Calcolo dell'età finale tramite Expected Value della Gaussiana combinata
            final_pred_age = (final_hybrid_prob @ bin_centers)[0]
            
            error = abs(final_pred_age - true_age_val)
            ensemble_errors.append(error)
            
            all_true_ages.append(true_age_val)
            all_ensemble_preds.append(final_pred_age)
            
    hybrid_mae = np.mean(ensemble_errors)
    print(f"\n>>> MAE HYBRID ENSEMBLE SUL TEST SET CATTIVO: {hybrid_mae:.2f} anni <<<")
    
    # --- PLOT RISULTATI ---
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    plt.scatter(all_true_ages, all_ensemble_preds, color='purple', edgecolor='white', s=90, alpha=0.9)
    min_val = min(min(all_true_ages), min(all_ensemble_preds)) - 2
    max_val = max(max(all_true_ages), max(all_ensemble_preds)) + 2
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2.5, label='Predizione Perfetta')
    plt.title('Hybrid Ensemble (1 FLAIR + 5 T1): Età Reale vs Predetta')
    plt.xlabel('Età Reale (Anni)')
    plt.ylabel('Età Predetta (Anni)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.subplot(1, 2, 2)
    plt.bar(range(1, len(ensemble_errors) + 1), ensemble_errors, color='gold', edgecolor='k', alpha=0.8)
    plt.axhline(y=hybrid_mae, color='k', linestyle='dashed', linewidth=2.5, label=f'MAE Medio: {hybrid_mae:.2f} anni')
    plt.title('Errore Assoluto per Paziente (Hybrid Ensemble)')
    plt.xlabel('Indice Paziente')
    plt.ylabel('Errore Assoluto (Anni Sbagliati)')
    plt.xticks(range(1, len(ensemble_errors) + 1))
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, '06_hybrid_multimodal_ensemble.png'))
    plt.show()
